In [1]:
# -*- coding: utf-8 -*-
"""ELT Pipeline - Acidentes PRF

Automatically generated by Colab.

# ELT: Extração → Carga (dados brutos) → Transformação (SQL no SQLite)

Diferença em relação ao ETL:
- No ETL: os dados são transformados em Python/Pandas antes de serem carregados.
- No ELT: os dados são carregados brutos no banco primeiro, e todas as
  transformações acontecem dentro do banco via SQL.
"""

import pandas as pd
import os
import sqlite3



# CONFIGURAÇÕES INICIAIS


In [12]:

# __file__ aponta para Notebooks/elt.py → dirname duas vezes = raiz do projeto
try:
    PASTA_TRABALHO = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    # ambiente Colab: notebook não tem __file__
    PASTA_TRABALHO = os.path.dirname(os.path.abspath(os.getcwd()))
print(PASTA_TRABALHO)
DB_PATH = os.path.join(PASTA_TRABALHO, "acidentes_prf.db")

/


# EXTRAÇÃO (Extract)
# Leitura dos arquivos CSV de cada ano — idêntica ao ETL


In [13]:


nomes_arquivos = ["2024", "2025", "2026"]
dataframes_por_ano = {}

for ano in nomes_arquivos:
    diretorio = os.path.join(PASTA_TRABALHO, "content/dataset", ano)
    for tabela in os.listdir(diretorio):
        caminho = os.path.join(diretorio, tabela)
        df = pd.read_csv(caminho, sep=";", encoding="latin1")
    dataframes_por_ano[ano] = df
    print(f"  [{ano}] {len(df):,} registros extraídos")

dataset_raw = pd.concat(dataframes_por_ano.values(), ignore_index=True)
print(f"\n  Total combinado: {len(dataset_raw):,} registros | {len(dataset_raw.columns)} colunas")


  [2024] 196,306 registros extraídos
  [2025] 194,629 registros extraídos
  [2026] 64,125 registros extraídos

  Total combinado: 455,060 registros | 35 colunas


# CARGA (Load) — dados brutos, sem nenhuma transformação
# Essa é a diferença central do ELT: o banco recebe os dados como vieram

In [14]:


conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Limpa tabelas de execuções anteriores para garantir idempotência
tabelas = [
    "fato_acidentes",
    "dim_pessoa", "dim_acidente", "dim_veiculo",
    "dim_local", "dim_clima", "dim_data", "dim_adm_prf",
    "acidentes_staging", "acidentes_raw",
]
for t in tabelas:
    cursor.execute(f"DROP TABLE IF EXISTS {t}")
conn.commit()

# Carrega os dados brutos diretamente — sem transformação alguma
dataset_raw.to_sql("acidentes_raw", conn, if_exists="replace", index=False)
n_raw = cursor.execute("SELECT COUNT(*) FROM acidentes_raw").fetchone()[0]
print(f"  Tabela 'acidentes_raw' criada: {n_raw:,} linhas, {len(dataset_raw.columns)} colunas")
print(f"  Banco de dados: {DB_PATH}")


  Tabela 'acidentes_raw' criada: 455,060 linhas, 35 colunas
  Banco de dados: /acidentes_prf.db


# TRANSFORMAÇÃO (Transform) — tudo feito via SQL dentro do SQLite

In [ ]:

print("\n  [3.1] Criando tabela 'acidentes_staging'...")

cursor.executescript("""
    CREATE TABLE acidentes_staging AS
    SELECT
        id,
        pesid,
        id_veiculo,

        -- datas e horários mantidos como texto (SQLite não tem tipo DATE/TIME nativo)
        -- a semântica de data é preservada pelo formato ISO YYYY-MM-DD
        data_inversa,
        horario,

        -- coordenadas e km: substituição de vírgula por ponto e conversão para REAL
        CAST(REPLACE(CAST(latitude  AS TEXT), ',', '.') AS REAL) AS latitude,
        CAST(REPLACE(CAST(longitude AS TEXT), ',', '.') AS REAL) AS longitude,
        CAST(REPLACE(CAST(km        AS TEXT), ',', '.') AS REAL) AS km,

        -- ano do veículo: valor 0 indica ausência de informação → NULL
        CASE
            WHEN CAST(ano_fabricacao_veiculo AS INTEGER) = 0 THEN NULL
            ELSE CAST(ano_fabricacao_veiculo AS INTEGER)
        END AS ano_fabricacao_veiculo,

        -- idade: -1 indica ausência de dado; valores > 127 são impossíveis → NULL
        CASE
            WHEN CAST(idade AS INTEGER) = -1  THEN NULL
            WHEN CAST(idade AS INTEGER) > 127 THEN NULL
            ELSE CAST(idade AS INTEGER)
        END AS idade,

        -- classificacao_acidente: NULL substituído por categoria prevista no dicionário
        COALESCE(classificacao_acidente, 'Ignorado') AS classificacao_acidente,

        -- tracado_via é multivalorada (ex: "Aclive;Curva"), não compatível com modelo estrela.
        -- Cada valor atômico vira uma coluna booleana (1 = presente, 0 = ausente).
        -- Padrões LIKE usam substrings ASCII para evitar problemas com acentuação.
        CASE WHEN tracado_via LIKE '%Aclive%'    THEN 1 ELSE 0 END AS trv_aclive,
        CASE WHEN tracado_via LIKE '%Curva%'     THEN 1 ELSE 0 END AS trv_curva,
        CASE WHEN tracado_via LIKE '%Declive%'   THEN 1 ELSE 0 END AS trv_declive,
        CASE WHEN tracado_via LIKE '%Desvio%'    THEN 1 ELSE 0 END AS trv_desvio_temporario,
        CASE WHEN tracado_via LIKE '%Em Obras%'  THEN 1 ELSE 0 END AS trv_em_obras,
        CASE WHEN tracado_via LIKE '%ntersec%'   THEN 1 ELSE 0 END AS trv_intersecao_vias,
        CASE WHEN tracado_via LIKE '%Ponte%'     THEN 1 ELSE 0 END AS trv_ponte,
        CASE WHEN tracado_via LIKE '%Reta%'      THEN 1 ELSE 0 END AS trv_reta,
        CASE WHEN tracado_via LIKE '%Retorno%'   THEN 1 ELSE 0 END AS trv_retorno_regulamentado,
        CASE WHEN tracado_via LIKE '%Rotat%'     THEN 1 ELSE 0 END AS trv_rotatoria,
        CASE WHEN tracado_via LIKE '%nel%'       THEN 1 ELSE 0 END AS trv_tunel,
        CASE WHEN tracado_via LIKE '%Viaduto%'   THEN 1 ELSE 0 END AS trv_viaduto,

        -- identificada: linhas onde estado_fisico, tipo_envolvido e sexo são todos NULL
        -- representam pessoas não identificadas nos registros da PRF
        CASE
            WHEN estado_fisico IS NULL AND tipo_envolvido IS NULL AND sexo IS NULL THEN 0
            ELSE 1
        END AS identificada,

        -- colunas mantidas sem transformação
        causa_acidente,
        tipo_acidente,
        uf,
        br,
        municipio,
        sentido_via,
        uso_solo,
        tipo_pista,
        fase_dia,
        condicao_metereologica,
        tipo_veiculo,
        marca,
        estado_fisico,
        tipo_envolvido,
        sexo,
        regional,
        delegacia,
        uop

        -- colunas removidas por redundância com estado_fisico:
        --   ilesos, feridos_leves, feridos_graves, mortos
        -- coluna removida por ter sido expandida nas booleanas trv_*:
        --   tracado_via

    FROM acidentes_raw;
""")
conn.commit()

n_staging = cursor.execute("SELECT COUNT(*) FROM acidentes_staging").fetchone()[0]
print(f"  Tabela 'acidentes_staging' criada: {n_staging:,} linhas")

# -----------------------------------------------------------------------------
# 3.2 Tabelas dimensão — criadas via SELECT DISTINCT + ROW_NUMBER como SK
# -----------------------------------------------------------------------------

print("\n  [3.2] Criando tabelas dimensão...")

cursor.executescript("""
    -- dim_pessoa: cada combinação única de atributos de pessoa
    CREATE TABLE dim_pessoa AS
    SELECT
        ROW_NUMBER() OVER () AS id_pessoa_sk,
        pesid,
        estado_fisico,
        tipo_envolvido,
        idade,
        sexo
    FROM (
        SELECT DISTINCT pesid, estado_fisico, tipo_envolvido, idade, sexo
        FROM acidentes_staging
    );

    -- dim_acidente: características de cada ocorrência de acidente
    CREATE TABLE dim_acidente AS
    SELECT
        ROW_NUMBER() OVER () AS id_acidente_sk,
        id,
        causa_acidente,
        tipo_acidente,
        classificacao_acidente
    FROM (
        SELECT DISTINCT id, causa_acidente, tipo_acidente, classificacao_acidente
        FROM acidentes_staging
    );

    -- dim_veiculo: características de cada veículo envolvido
    CREATE TABLE dim_veiculo AS
    SELECT
        ROW_NUMBER() OVER () AS id_veiculo_sk,
        id_veiculo,
        tipo_veiculo,
        marca,
        ano_fabricacao_veiculo
    FROM (
        SELECT DISTINCT id_veiculo, tipo_veiculo, marca, ano_fabricacao_veiculo
        FROM acidentes_staging
    );

    -- dim_local: localização e características da via
    CREATE TABLE dim_local AS
    SELECT
        ROW_NUMBER() OVER () AS id_local_sk,
        uf, br, km, municipio, latitude, longitude,
        sentido_via, uso_solo, tipo_pista,
        trv_aclive, trv_curva, trv_declive, trv_desvio_temporario,
        trv_em_obras, trv_intersecao_vias, trv_ponte, trv_reta,
        trv_retorno_regulamentado, trv_rotatoria, trv_tunel, trv_viaduto
    FROM (
        SELECT DISTINCT
            uf, br, km, municipio, latitude, longitude,
            sentido_via, uso_solo, tipo_pista,
            trv_aclive, trv_curva, trv_declive, trv_desvio_temporario,
            trv_em_obras, trv_intersecao_vias, trv_ponte, trv_reta,
            trv_retorno_regulamentado, trv_rotatoria, trv_tunel, trv_viaduto
        FROM acidentes_staging
    );

    -- dim_clima: condições ambientais no momento do acidente
    CREATE TABLE dim_clima AS
    SELECT
        ROW_NUMBER() OVER () AS id_clima_sk,
        fase_dia,
        condicao_metereologica
    FROM (
        SELECT DISTINCT fase_dia, condicao_metereologica
        FROM acidentes_staging
    );

    -- dim_data: dimensão temporal com atributos calculados via SQL
    CREATE TABLE dim_data AS
    SELECT
        ROW_NUMBER() OVER () AS id_data_sk,
        data_inversa,
        horario,
        CAST(SUBSTR(data_inversa, 9, 2) AS INTEGER) AS dia,
        CAST(SUBSTR(data_inversa, 6, 2) AS INTEGER) AS mes,
        CAST(SUBSTR(data_inversa, 1, 4) AS INTEGER) AS ano,
        CASE CAST(SUBSTR(data_inversa, 6, 2) AS INTEGER)
            WHEN 1 THEN 1  WHEN 2 THEN 1  WHEN 3 THEN 1
            WHEN 4 THEN 2  WHEN 5 THEN 2  WHEN 6 THEN 2
            WHEN 7 THEN 3  WHEN 8 THEN 3  WHEN 9 THEN 3
            ELSE 4
        END AS trimestre,
        CASE CAST(STRFTIME('%w', data_inversa) AS INTEGER)
            WHEN 0 THEN 'Sunday'    WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'   WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'  WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
        END AS dia_semana
    FROM (
        SELECT DISTINCT data_inversa, horario
        FROM acidentes_staging
        ORDER BY data_inversa, horario
    );

    -- dim_adm_prf: unidades administrativas da PRF responsáveis
    CREATE TABLE dim_adm_prf AS
    SELECT
        ROW_NUMBER() OVER () AS id_adm_prf_sk,
        regional,
        delegacia,
        uop
    FROM (
        SELECT DISTINCT regional, delegacia, uop
        FROM acidentes_staging
    );
""")
conn.commit()

dims = {
    "dim_pessoa": "pessoas",
    "dim_acidente": "acidentes",
    "dim_veiculo": "veículos",
    "dim_local": "locais",
    "dim_clima": "climas",
    "dim_data": "datas",
    "dim_adm_prf": "unidades adm.",
}
for tabela, descricao in dims.items():
    n = cursor.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"    {tabela:<18} {n:>8,} {descricao}")

# -----------------------------------------------------------------------------
# 3.3 Tabela fato — une staging com todas as dimensões via chaves naturais
# JOINs NULL-safe: (a = b OR (a IS NULL AND b IS NULL))
# -----------------------------------------------------------------------------

print("\n  [3.3] Criando tabela 'fato_acidentes'...")

cursor.executescript("""
    CREATE TABLE fato_acidentes AS
    SELECT
        p.id_pessoa_sk,
        a.id_acidente_sk,
        v.id_veiculo_sk,
        l.id_local_sk,
        c.id_clima_sk,
        d.id_data_sk,
        adm.id_adm_prf_sk,
        s.identificada
    FROM acidentes_staging s

    LEFT JOIN dim_pessoa p
        ON  s.pesid = p.pesid
        AND (s.estado_fisico   = p.estado_fisico   OR (s.estado_fisico   IS NULL AND p.estado_fisico   IS NULL))
        AND (s.tipo_envolvido  = p.tipo_envolvido  OR (s.tipo_envolvido  IS NULL AND p.tipo_envolvido  IS NULL))
        AND (s.idade           = p.idade           OR (s.idade           IS NULL AND p.idade           IS NULL))
        AND (s.sexo            = p.sexo            OR (s.sexo            IS NULL AND p.sexo            IS NULL))

    LEFT JOIN dim_acidente a
        ON  s.id = a.id
        AND s.causa_acidente         = a.causa_acidente
        AND s.tipo_acidente          = a.tipo_acidente
        AND s.classificacao_acidente = a.classificacao_acidente

    LEFT JOIN dim_veiculo v
        ON  s.id_veiculo = v.id_veiculo
        AND (s.tipo_veiculo          = v.tipo_veiculo          OR (s.tipo_veiculo          IS NULL AND v.tipo_veiculo          IS NULL))
        AND (s.marca                 = v.marca                 OR (s.marca                 IS NULL AND v.marca                 IS NULL))
        AND (s.ano_fabricacao_veiculo= v.ano_fabricacao_veiculo OR (s.ano_fabricacao_veiculo IS NULL AND v.ano_fabricacao_veiculo IS NULL))

    LEFT JOIN dim_local l
        ON  (s.uf        = l.uf        OR (s.uf        IS NULL AND l.uf        IS NULL))
        AND (s.br        = l.br        OR (s.br        IS NULL AND l.br        IS NULL))
        AND (s.km        = l.km        OR (s.km        IS NULL AND l.km        IS NULL))
        AND (s.municipio = l.municipio OR (s.municipio IS NULL AND l.municipio IS NULL))
        AND (s.latitude  = l.latitude  OR (s.latitude  IS NULL AND l.latitude  IS NULL))
        AND (s.longitude = l.longitude OR (s.longitude IS NULL AND l.longitude IS NULL))
        AND (s.sentido_via = l.sentido_via OR (s.sentido_via IS NULL AND l.sentido_via IS NULL))
        AND (s.uso_solo    = l.uso_solo    OR (s.uso_solo    IS NULL AND l.uso_solo    IS NULL))
        AND (s.tipo_pista  = l.tipo_pista  OR (s.tipo_pista  IS NULL AND l.tipo_pista  IS NULL))
        AND s.trv_aclive                = l.trv_aclive
        AND s.trv_curva                 = l.trv_curva
        AND s.trv_declive               = l.trv_declive
        AND s.trv_desvio_temporario     = l.trv_desvio_temporario
        AND s.trv_em_obras              = l.trv_em_obras
        AND s.trv_intersecao_vias       = l.trv_intersecao_vias
        AND s.trv_ponte                 = l.trv_ponte
        AND s.trv_reta                  = l.trv_reta
        AND s.trv_retorno_regulamentado = l.trv_retorno_regulamentado
        AND s.trv_rotatoria             = l.trv_rotatoria
        AND s.trv_tunel                 = l.trv_tunel
        AND s.trv_viaduto               = l.trv_viaduto

    LEFT JOIN dim_clima c
        ON  (s.fase_dia              = c.fase_dia              OR (s.fase_dia              IS NULL AND c.fase_dia              IS NULL))
        AND (s.condicao_metereologica= c.condicao_metereologica OR (s.condicao_metereologica IS NULL AND c.condicao_metereologica IS NULL))

    LEFT JOIN dim_data d
        ON  s.data_inversa = d.data_inversa
        AND s.horario      = d.horario

    LEFT JOIN dim_adm_prf adm
        ON  (s.regional  = adm.regional  OR (s.regional  IS NULL AND adm.regional  IS NULL))
        AND (s.delegacia = adm.delegacia OR (s.delegacia IS NULL AND adm.delegacia IS NULL))
        AND (s.uop       = adm.uop       OR (s.uop       IS NULL AND adm.uop       IS NULL));
""")
conn.commit()

n_fato = cursor.execute("SELECT COUNT(*) FROM fato_acidentes").fetchone()[0]
print(f"  Tabela 'fato_acidentes' criada: {n_fato:,} linhas")



  [3.1] Criando tabela 'acidentes_staging'...
  Tabela 'acidentes_staging' criada: 455,060 linhas

  [3.2] Criando tabelas dimensão...
    dim_pessoa          415,119 pessoas
    dim_acidente        169,160 acidentes
    dim_veiculo         324,360 veículos
    dim_local           159,996 locais
    dim_clima                36 climas
    dim_data            113,337 datas
    dim_adm_prf             454 unidades adm.

  [3.3] Criando tabela 'fato_acidentes'...


In [6]:
# =============================================================================
# RESUMO FINAL
# =============================================================================

print("\n" + "=" * 60)
print("ELT CONCLUÍDO")
print("=" * 60)
print(f"\n  Banco de dados: {DB_PATH}")
print(f"\n  Tabelas criadas:")
todas_tabelas = tabelas[::-1]  # ordem de criação
for t in ["acidentes_raw", "acidentes_staging",
          "dim_pessoa", "dim_acidente", "dim_veiculo",
          "dim_local", "dim_clima", "dim_data", "dim_adm_prf",
          "fato_acidentes"]:
    n = cursor.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"    {t:<25} {n:>10,} linhas")

conn.close()


ELT CONCLUÍDO

  Banco de dados: /acidentes_prf.db

  Tabelas criadas:


OperationalError: no such table: acidentes_raw